In [232]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import fsolve
from matplotlib.lines import Line2D

T = 329.15  # K  (56°C)
antoine = {
    "EtOH":  (5.37229, 1670.409, -40.191),
    "EtOAc": (4.22809, 1245.702, -55.189),
    "MeOAc": (4.20364, 1164.426, -52.690),
}

def Psat(comp, T):
    A, B, C = antoine[comp]
    return 10**(A - B / (T + C))   # bar
Psat_vals = {k: Psat(k, T) for k in antoine}

In [233]:
def gamma_margules(x1, a12, a21):
    x2 = 1 - x1
    lng1 = (a12 + 2*(a21 - a12)*x1) * x2**2
    lng2 = (a21 + 2*(a12 - a21)*x2) * x1**2
    return np.exp(lng1), np.exp(lng2)

def gamma_vanlaar(x1, alpha, beta, EPS = 1e-12):
    x2 = 1 - x1
    # guard x1=0 and x2=0 edges
    with np.errstate(divide='ignore', invalid='ignore'):
        denom1 = np.where(x2 < EPS, 1.0,
                          (1 + (alpha/beta) * (x1 / np.maximum(x2, EPS)))**2)
        denom2 = np.where(x1 < EPS, 1.0,
                          (1 + (beta/alpha) * (x2 / np.maximum(x1, EPS)))**2)
    g1 = np.where(x2 < EPS, 1.0, np.exp(alpha / denom1))
    g2 = np.where(x1 < EPS, 1.0, np.exp(beta  / denom2))
    return g1, g2

def gamma_wilson(x1, L12, L21, EPS = 1e-12):
    x2 = 1 - x1
    A = x1 + x2*L12
    B = x1*L21 + x2
    bracket = L12/A - L21/B
    lng1 = -np.log(np.maximum(A, EPS)) + x2 * bracket
    lng2 = -np.log(np.maximum(B, EPS)) - x1 * bracket
    return np.exp(lng1), np.exp(lng2)

In [234]:
def compute_Pxy(P1s, P2s, gamma_fn, params):
    """Return (x1, y1, P) arrays across full composition range."""
    x1 = np.linspace(0, 1, 300)
    g1, g2 = gamma_fn(x1, *params)
    P  = x1*g1*P1s + (1-x1)*g2*P2s
    y1 = np.where(P > 0, x1*g1*P1s / P, x1)
    return x1, y1, P

# ── Parameter extraction
def extract_margules(gi1, gi2):
    return np.log(gi1), np.log(gi2)

def extract_vanlaar(gi1, gi2):
    return np.log(gi1), np.log(gi2)

def extract_wilson(gi1, gi2):
    lnG1, lnG2 = np.log(gi1), np.log(gi2)
    def eqs(p):
        L12, L21 = p
        return [-np.log(L12) + 1 - L21 - lnG1,
                -np.log(L21) + 1 - L12 - lnG2]
    best = None
    for L12_0, L21_0 in [(0.5,0.5),(0.3,0.7),(0.7,0.3),
                          (0.1,0.9),(0.9,0.1),(0.4,1.5),(1.5,0.4)]:
        try:
            x, _, ier, _ = fsolve(eqs, [L12_0, L21_0], full_output=True)
            L12, L21 = x
            res = max(abs(r) for r in eqs(x))
            if ier==1 and L12>1e-9 and L21>1e-9 and res<1e-10:
                if best is None or res < best[2]:
                    best = (L12, L21, res)
        except Exception:
            pass
    return (best[0], best[1]) if best else (None, None)

In [235]:
import pandas as pd

systems = [
    {
        "label":  "EtOH (1) – MeOAc (2)",
        "comp1":  "EtOH",
        "comp2":  "MeOAc",
        "x1label": "$x_1, y_1$  [EtOH]",
        "gi_exp": (2.538, 3.863),
        "gi_asp": (2.728, 3.153),
        "ylims": (0.0, 1.2),
    },
    {
        "label":  "MeOAc (1) – EtOAc (2)",
        "comp1":  "MeOAc",
        "comp2":  "EtOAc",
        "x1label": "$x_1, y_1$  [MeOAc]",
        "gi_exp": (1.166, 1.042),
        "gi_asp": (1.230, 1.110),
        "ylims": (0.0, 1.2),
    },
    {
        "label":  "EtOAc (1) – EtOH (2)",
        "comp1":  "EtOAc",
        "comp2":  "EtOH",
        "x1label": "$x_1, y_1$  [EtOAc]",
        "gi_exp": (3.400, 2.639),
        "gi_asp": (2.625, 2.489),
        "ylims": (0.3, 0.6),
    },
]

In [236]:
# # Extract parameters for each system and model

models = [
    ("Margules",  extract_margules, gamma_margules),
    ("van Laar",  extract_vanlaar,  gamma_vanlaar),
    ("Wilson",    extract_wilson,   gamma_wilson),
]

# results = []
# for sys in systems:
#     # for src_label, src in [("Experimental", "gi_exp")]:
#     for src_label, src in [("Aspen", "gi_asp")]:
#         gi1, gi2 = sys[src]
#         for mname, extfn, _ in models:
#             params = extfn(gi1, gi2)
#             if params is None or any(p is None for p in params):
#                 continue
#             entry = {"System": sys["label"], "Source": src_label, "Model": mname}
#             if mname == "Margules":
#                 entry.update({"a12": params[0], "a21": params[1]})
#             elif mname == "van Laar":
#                 entry.update({"alpha": params[0], "beta": params[1]})
#             elif mname == "Wilson":
#                 entry.update({"L12": params[0], "L21": params[1]})
#             results.append(entry)

# df_results = pd.DataFrame(results)
# df_results.to_clipboard()
# df_results

In [237]:
import matplotlib.patches as mpatches
def add_row_bands(fig, axes, band_cfg=None, band_height=0.03, pad=0.034):
    """Add labeled bands above rows of axes."""
    def row_bbox(axes_row):
        """Return (x_left, x_right, y_bottom, y_top) in figure coords for a row."""
        bboxes = [ax.get_position() for ax in axes_row]
        x0 = min(b.x0 for b in bboxes)
        x1 = max(b.x1 for b in bboxes)
        y0 = min(b.y0 for b in bboxes)
        y1 = max(b.y1 for b in bboxes)
        return x0, x1, y0, y1


    if band_cfg is None:
        band_cfg = [
            (axes[0], "Experimental", "#ddeeff", "#0C447C"),
            (axes[1], "Aspen",        "#fef3dc", "#633806"),
        ]

    for ax_row, label, fc, tc in band_cfg:
        x0, x1, y0, y1 = row_bbox(ax_row)
        band_y = y1 + pad

        rect = mpatches.FancyBboxPatch(
            (x0, band_y),
            x1 - x0, band_height,
            boxstyle="square,pad=0",
            transform=fig.transFigure,
            facecolor=fc,
            edgecolor="none",
            clip_on=False,
            zorder=3,
        )
        fig.add_artist(rect)

        fig.text(
            0.5, band_y + band_height / 2,
            label,
            transform=fig.transFigure,
            ha="center", va="center",
            fontsize=10, fontweight="bold",
            color=tc,
            zorder=4,
        )

In [238]:
def plot_models_to_file(models, outname, figsize=(13,3), ncolumns=5):
    """Generic plot routine for one or more thermodynamic models.

    models: list of (name, extract_fn, gamma_fn)
    outname: output filename (PDF)
    """
    model_colors = {
        "Margules": "#e41a1c",
        "van Laar": "#377eb8",
        "Wilson":   "#4daf4a",
    }

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(1, 3, figure=fig, hspace=0.38, wspace=0.32)

    sources    = ["gi_exp", "gi_asp"]
    source_labels = ["Experimental", "Aspen"]

    axes = [fig.add_subplot(gs[0, c]) for c in range(3)]

    for col, sys in enumerate(systems):
        ax = axes[col]
        P1s = Psat_vals[sys["comp1"]]
        P2s = Psat_vals[sys["comp2"]]
        
        for src_idx, (src, src_label) in enumerate(zip(sources, source_labels)):
            gi1, gi2 = sys[src]

            for mname, extfn, gammafn in models:
                params = extfn(gi1, gi2)
                if params is None or any(p is None for p in params):
                    continue
                x1, y1, P = compute_Pxy(P1s, P2s, gammafn, params)
                P_mbar = P

                lw = 1.8
                # Experimental uses model color, Aspen uses black
                color = model_colors[mname] if src_label == "Experimental" else "#434343"
                
                ax.plot(x1, P_mbar, color=color,
                        lw=lw, label=f"{mname} ({src_label})")
                ax.plot(y1, P_mbar, color=color,
                        lw=lw, ls="dashdot")

        ax.set_xlabel(sys["x1label"], fontsize=9)
        ax.set_xlim(0, 1)
        ylims = sys.get("ylims")
        if ylims is not None:
            ax.set_ylim(ylims)
        ax.set_ylabel("$P$ (bar)", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(alpha=0.2)
        ax.set_title(sys["label"], fontsize=10, fontweight="bold", pad=6)

    # legend: include only models actually plotted
    legend_handles = []
    for mname in model_colors:
        if any(m[0] == mname for m in models):
            legend_handles.append(Line2D([0],[0], color=model_colors[mname], lw=2, label=mname))
    legend_handles.extend([
        Line2D([0],[0], color="k", lw=2, label="Aspen"),
        Line2D([0],[0], color="k", lw=1.5, ls="-",  label="liquid ($x_1$)"),
        Line2D([0],[0], color="k", lw=1.5, ls="dashdot", label="vapor  ($y_1$)"),
    ])
    fig.legend(handles=legend_handles, loc="lower center", ncol=ncolumns,
               fontsize=13, frameon=True, bbox_to_anchor=(0.50, -0.20))

    plt.savefig(outname, dpi=300, bbox_inches="tight")
    plt.close(fig)

# Wilson

In [239]:
models = [
    ("Wilson",    extract_wilson,   gamma_wilson),
]

plot_models_to_file(models, "wilsonPxy.pdf", ncolumns=5)


/var/folders/f9/38nkw6517lgf362dv55hy0xc0000gn/T/ipykernel_91648/4110884564.py:21: RuntimeWarning: invalid value encountered in log
  -np.log(L21) + 1 - L12 - lnG2]
/var/folders/f9/38nkw6517lgf362dv55hy0xc0000gn/T/ipykernel_91648/4110884564.py:20: RuntimeWarning: invalid value encountered in log
  return [-np.log(L12) + 1 - L21 - lnG1,


# van Laar

In [240]:
models = [
    ("van Laar",  extract_vanlaar,  gamma_vanlaar),
]

plot_models_to_file(models, "vanLaarPxy.pdf")

# all

In [241]:
models = [
    ("Margules",  extract_margules, gamma_margules),]
plot_models_to_file(models, "margulesPxy.pdf")